<style>
div.mermaid > svg { width: 70% !important; height: auto !important; }
</style>

# Blackwell tcgen05 GEMM

This is a complete warp-specialized `C = A @ B` for Blackwell's **5th-gen tensor cores**
(`tcgen05`). One CTA computes a single 128x128 output tile: a **TMA warp** streams the A and B
tiles from global memory into shared memory, an **MMA warp** drives `tcgen05_mma` into **Tensor
Memory (TMEM)**, and four **epilogue warps** read the result back out to C.

The GEMM is **fp16 in, fp16 out, fp32 accumulate**: both A and B operands are `Float16`, the MMA
accumulates in `Float32` (in TMEM), and the epilogue casts the fp32 result back down to `Float16`
for C.

The `tcgen05`/mbarrier intrinsics used here (the `prims.*` calls) are the **NVVM-level interface**
-- thin wrappers over the NVVM ops that drive the tensor cores, TMA engine, and mbarriers
directly. This notebook is also a tour of that interface.

It is the smallest kernel that exercises the whole Blackwell GEMM pipeline -- TMA, mbarriers,
`tcgen05` MMA, and TMEM -- and the `04_blackwell_mma_contextvar` notebook scales exactly this shape
into a multi-CTA M x N x K GEMM.

**You'll learn:** the three warp roles of a tcgen05 GEMM (TMA load, MMA, epilogue) and the two
mbarriers that chain them; how the result lands in **Tensor Memory (TMEM)** and how the epilogue
reads it back; and the **SMEM and instruction descriptors** that tell `tcgen05_mma` where its
operands sit and how to accumulate. **Prereq:** the `04_tiled_gemm` notebook (tiled GEMM, shared
memory, CTA barriers) and the `05_tma_load` notebook (TMA + mbarriers).

**Runs on:** datacenter Blackwell -- `sm_100a` (B100/B200) and `sm_100f` (B300). `tcgen05` and TMEM
do not exist on Hopper or consumer `sm_120`. No GPU? It compiles without a GPU via
`CUTE_DSL_DRYRUN=1`.

In [ ]:
import cutlass  # Array, Float16/Float32, Int32/64, Constexpr, GridConstant, AddressSpace
import cutlass.cute as cute  # @cute.kernel / @cute.jit, cute.arch.*, cute.runtime
from cutlass.experimental import primitives as prims  # tcgen05 / mbarrier / TMA / barrier intrinsics + SMEM/Instr descriptors, TmemAddr
import cutlass.experimental.cuda as cuda  # TMA descriptors: create_tensor_map_*, TensorMap(Swizzle)

import torch
from typing import List

## 1. The warp-specialized skeleton

Three warp roles split the work, chained by two mbarriers:

```text
  warp 0      TMA warp      : copies A and B global -> SMEM, signals mbar_tma (operands ready)
  warp 1      MMA warp      : drives tcgen05_mma into Tensor Memory (TMEM), signals mbar_mma (result ready)
  warps 2,3   (exit)        : skipped, so the four epilogue warps form one clean warp-group
  warps 4-7   epilogue      : read the 128x128 result from TMEM and write it to C
```

The result lands in **Tensor Memory (TMEM)** -- a Blackwell-only on-chip store that `tcgen05_mma`
writes and the epilogue reads back. The MMA reads its operands out of SMEM through two **SMEM
descriptors** (`Tcgen05SmemDesc`) whose swizzle and strides must match the TMA descriptor the host
builds. A third **instruction descriptor** (`Tcgen05InstrDesc`) sets the accumulation type: fp16
operands accumulate in fp32, which is why the epilogue reads fp32 out of TMEM and casts down to
fp16 for C.

In [ ]:
# =============================================================================
# Kernel: a warp-specialized tcgen05 GEMM for one 128x128 tile.
#         TMA warp loads, MMA warp runs tcgen05_mma into TMEM, epilogue reads it.
# =============================================================================
@cute.kernel
def gemm_kernel(
    tma_desc_a: cutlass.GridConstant[cuda.TensorMap],
    tma_desc_b: cutlass.GridConstant[cuda.TensorMap],
    matrix_c: cutlass.Array,
    problem_size: cutlass.Constexpr[List[int]],
) -> None:
    M, K, N = problem_size
    thread_id, _, _ = cute.arch.thread_idx()
    warp_id = cute.arch.warp_idx()
    tmem_cols = N  # the TMEM result tile is N columns wide

    # Step 1. Allocate the SMEM operand tiles, the two stage mbarriers, and a
    # slot for the TMEM pointer.
    smem_a = cutlass.Array(cutlass.Float16, (M, K), space=cutlass.AddressSpace.smem)
    smem_b = cutlass.Array(cutlass.Float16, (N, K), space=cutlass.AddressSpace.smem)
    mbar_tma = cutlass.Array(cutlass.Int64, 1, space=cutlass.AddressSpace.smem)
    mbar_mma = cutlass.Array(cutlass.Int64, 1, space=cutlass.AddressSpace.smem)
    tmem_ptr_i32 = cutlass.Array(cutlass.Int32, 1, space=cutlass.AddressSpace.smem)

    # Step 2. One elected thread prefetches the TMA descriptors and arms both
    # mbarriers (each expects a single arrival); the whole CTA syncs before use.
    if prims.elect_sync():
        prims.prefetch_tensormap(tma_desc_a.get_ptr())
        prims.prefetch_tensormap(tma_desc_b.get_ptr())
        prims.mbarrier_init(mbar_tma, 1)
        prims.mbarrier_init(mbar_mma, 1)

    prims.fence_mbarrier_init()
    cute.arch.barrier()

    # Step 3. Assign warp roles. Warps 2-3 are unused; exit them so the epilogue
    # warp-group (4-7) lines up cleanly.
    is_tma_warp = warp_id == 0
    is_tc_warp = warp_id == 1
    is_epi_warp = warp_id > 3
    if warp_id == 2 or warp_id == 3:
        prims.exit()

    # Step 4. The MMA warp owns the TMEM allocation; the result tile lives there
    # until the epilogue drains it. Everyone reads the pointer after the alloc syncs.
    if is_tc_warp:
        prims.tcgen05_alloc(tmem_ptr_i32, tmem_cols)
    cute.arch.barrier()
    tmem_ptr = prims.make_tmem_ptr(tmem_ptr_i32.load(), cutlass.Int32)

    # Step 5. Run the warp-specialized body: TMA load, then MMA, then epilogue.
    if is_tma_warp:
        # Producer: trim the register file (light work), then have one thread
        # post the expected byte count and kick off both TMA copies. The copies
        # signal mbar_tma when the bytes land.
        prims.setmaxregister(40, prims.SetMaxRegisterAction.DECREASE)
        prims.bar_warp_sync(cute.arch.FULL_MASK)
        if prims.elect_sync():
            size_a = (K * M) * cutlass.Float16.width // 8
            size_b = (K * N) * cutlass.Float16.width // 8
            prims.mbarrier_arrive_expect_tx(mbar_tma, size_a + size_b)
            prims.cp_async_bulk_tensor_shared_cta_global(
                smem_a, tma_desc_a.get_ptr(), (0, 0), mbar_tma
            )
            prims.cp_async_bulk_tensor_shared_cta_global(
                smem_b, tma_desc_b.get_ptr(), (0, 0), mbar_tma
            )

    elif is_tc_warp:
        # Consumer: wait for the operands, then run tcgen05_mma into TMEM.
        prims.bar_warp_sync(cute.arch.FULL_MASK)
        if prims.elect_sync():
            while not prims.mbarrier_try_wait_parity(mbar_tma, 0):
                pass
            # SMEM descriptors describe where/how the operands sit in SMEM; the
            # swizzle/strides must match what the host's TMA descriptor wrote
            # (s128b -> stride_byte_offset 1024, leading_byte_offset 16).
            desc_a = prims.Tcgen05SmemDesc.build(
                smem_a,
                leading_byte_offset=16,
                stride_byte_offset=1024,
                layout=prims.Tcgen05SmemSwizzle.SWIZZLE_128B,
            )
            desc_b = prims.Tcgen05SmemDesc.build(
                smem_b,
                leading_byte_offset=16,
                stride_byte_offset=1024,
                layout=prims.Tcgen05SmemSwizzle.SWIZZLE_128B,
            )
            # Instruction descriptor: fp16 operands accumulate in fp32.
            idesc = prims.Tcgen05InstrDesc.build(
                c_dtype=cutlass.Float32, n_dim=128, m_dim=128
            )
            # fp16 tensor-core K is 16, so K // 16 steps cover the K tile, each
            # advancing the descriptors by 16 elements * 2 bytes. scale_d=False on
            # the first step overwrites TMEM (no memset); True afterwards accumulates.
            scale_d = False
            for i in cutlass.range_constexpr(K // 16):
                off = 16 * 2 * i
                prims.tcgen05_mma(
                    prims.Tcgen05MMAKind.F16,
                    prims.CTAGroup.CTA_1,
                    tmem_ptr,
                    desc_a.advance_start_address(off),
                    desc_b.advance_start_address(off),
                    idesc,
                    scale_d,
                )
                scale_d = True
            prims.tcgen05_commit(mbar_mma)  # signal the result is ready

    elif is_epi_warp:
        # Epilogue: wait for the result, then drain TMEM -> C. Each of the four
        # warps owns a 32-row band of the 128-row tile. fp16 accumulated in fp32,
        # so TMEM holds one fp32 per column; read two columns per thread and cast
        # each down to fp16 for C.
        while not prims.mbarrier_try_wait_parity(mbar_mma, 0):
            pass
        warpid_in_epi_wg = warp_id % 4
        m_id = thread_id % 128
        tmem_base = prims.TmemAddr(tmem_ptr_i32.load())
        row_id = tmem_base.row_id + warpid_in_epi_wg * 32
        for n in range(0, tmem_cols, 2):
            tmem = prims.TmemAddr.from_row_col(
                row_id, tmem_base.col_id + n
            ).as_ptr(cutlass.Float32)
            c_rmem = prims.tcgen05_ld("32x32b", tmem, num=2)
            for i in cutlass.range_constexpr(2):
                matrix_c[m_id, n + i] = cutlass.Float16(c_rmem[i])

    cute.arch.barrier()

    # Step 6. Free the TMEM and release the allocation permit (MMA warp only).
    if is_tc_warp:
        prims.tcgen05_dealloc(tmem_ptr, tmem_cols)
        prims.tcgen05_relinquish_alloc_permit()

## 2. Host: build the TMA descriptors and launch

The host builds one tile-sized TMA descriptor per operand and launches a 256-thread (8-warp)
block. The descriptor **swizzle** must match the SMEM layout the kernel's `Tcgen05SmemDesc`
expects -- `s128b`, the 128-byte swizzle, paired with `stride_byte_offset=1024` in the kernel. A
flat `cutlass.Array` (what `from_dlpack` produces) feeds `create_tensor_map_tiled_from_view`
directly: it reads the source's shape, strides, and dtype to program the engine.

In [ ]:
# =============================================================================
# Host: build the A/B TMA descriptors and launch.
# =============================================================================
@cute.jit
def gemm(
    matrix_a: cutlass.Array,
    matrix_b: cutlass.Array,
    matrix_c: cutlass.Array,
    problem_size: cutlass.Constexpr[List[int]],
) -> None:
    M, K, N = problem_size
    # The TMA swizzle must match the kernel's SMEM layout (s128b). C is the
    # epilogue output and stays a plain cutlass.Array (no descriptor).
    tma_desc_a = cuda.create_tensor_map_tiled_from_view(
        matrix_a, box_dims=(M, K), swizzle=cuda.TensorMapSwizzle.s128b
    )
    tma_desc_b = cuda.create_tensor_map_tiled_from_view(
        matrix_b, box_dims=(N, K), swizzle=cuda.TensorMapSwizzle.s128b
    )
    # 256 threads = 8 warps, the split the kernel's warp roles assume.
    gemm_kernel(tma_desc_a, tma_desc_b, matrix_c, problem_size).launch(
        grid=(1, 1, 1), block=(256, 1, 1)
    )

## 3. Run it and check against PyTorch

We multiply one 128x128 tile with K=64. A is `(M, K)` row-major; B is stored `(N, K)` (also
K-major), so the reference transposes it to form the `(M, K) @ (K, N)` product. The PyTorch
reference is computed on the **CPU**, so the check passes identically whether the kernel ran on real
hardware or compiled under dryrun.

In [ ]:
# =============================================================================
# Main: run the GEMM and check against PyTorch.
# =============================================================================
# One 128x128 output tile with K=64.
M, N, K = 128, 128, 64
a = torch.randn(M, K, dtype=torch.float16, device="cuda")
b = torch.randn(N, K, dtype=torch.float16, device="cuda")  # logical (N, K), K-major rows
c = torch.zeros(M, N, dtype=torch.float16, device="cuda")

gemm(
    cute.runtime.from_dlpack(a),
    cute.runtime.from_dlpack(b),
    cute.runtime.from_dlpack(c),
    (M, K, N),
)

# CPU reference: B is stored (N, K), so transpose it to form (M, K) @ (K, N).
ref = a.cpu().float() @ b.cpu().float().T
torch.testing.assert_close(c.cpu().float(), ref, atol=1e-3, rtol=1e-3)
print("PASS")

# Expected output:
# PASS

## Try it yourself

1. **The two mbarriers.** `mbar_tma` is signalled by the TMA copies (it counts *bytes*) and
   `mbar_mma` by `tcgen05_commit` (it counts the MMA's completion). Trace which warp waits on which,
   and why the epilogue can't read TMEM until `mbar_mma` fires.
2. **`scale_d`.** The first `tcgen05_mma` uses `scale_d=False` (overwrite TMEM), the rest `True`
   (accumulate). What would change if every step used `True`, and why does "first write overwrites"
   save a TMEM memset?
3. **The accumulator dtype.** The instruction descriptor sets `c_dtype=Float32`, so the epilogue
   reads fp32 from TMEM and casts to fp16. Why must the read-back dtype match the accumulator the
   MMA wrote?
4. **Descriptor swizzle.** The kernel's `Tcgen05SmemDesc` uses `SWIZZLE_128B` with
   `stride_byte_offset=1024`, matching the host's `s128b`. What breaks if the host built the
   descriptor with `swizzle=none` instead?